# Homomorphic mode recovery via kernel: per-candidate traversal

Implements Sections 4.1 and 4.7 of the dissertation: a single packed ciphertext,
padding to `N = 2^ceil(log2 n)`, and the mask applied to the **output** of the
polynomial.

Contents:

1. Data and plaintext reference functions
2. Plaintext sweep over `(sigma, d)`, measuring `eps`, `Delta_S` and the condition `n*eps < Delta_S/2`
3. FHE run with the chosen parameters
4. Exact / approximate / FHE comparison
5. Block packing (Proposition 4.27)

Cells 1 and 2 run in seconds and do not need OpenFHE. Use them to choose the
parameters before spending time on FHE.

In [ ]:
import math
import numpy as np
from numpy.polynomial.chebyshev import Chebyshev

# ============================================================
# DATA
# ============================================================
dados = np.array(
    [10.0] * 20                                                    # mode: 20 copies
    + [20.03, 19.97, 20.08, 19.92, 20.12, 19.88,
       20.05, 19.95, 20.10, 19.90, 20.02]                          # cluster ~20
    + [29.94, 30.11, 29.89, 30.04, 29.96]                          # cluster ~30
)

n = len(dados)
N = 2 ** math.ceil(math.log2(n))

valores = np.unique(dados)
freqs = {v: int((dados == v).sum()) for v in valores}
v_estrela = max(freqs, key=freqs.get)
f_m = freqs[v_estrela]
delta_f = f_m - max(f for v, f in freqs.items() if v != v_estrela)
delta_min = min(abs(a - b) for a in valores for b in valores if a != b)
delta_max = dados.max() - dados.min()

print(f"n = {n},  N = {N},  valores distintos = {len(valores)}")
print(f"moda v* = {v_estrela},  f_m = {f_m},  Delta_f = {delta_f}")
print(f"Delta_min = {delta_min:.4f},  Delta_max = {delta_max:.4f}")
print(f"racio Delta_max/Delta_min = {delta_max/delta_min:.0f}")


# ============================================================
# REFERENCE FUNCTIONS (plaintext)
# ============================================================
def matriz_u(x, sigma):
    """u_ij = ((x_i - x_j)/sigma)^2"""
    return ((x[:, None] - x[None, :]) / sigma) ** 2


def scores_exatos(x, sigma):
    """S_i = sum_j K((x_i-x_j)/sigma),  K(t) = 1/(1+t^2)"""
    return (1.0 / (1.0 + matriz_u(x, sigma))).sum(axis=1)


def polinomio(R, d):
    """Chebyshev approximation of F(u) = 1/(1+u) on [0, R]."""
    return Chebyshev.interpolate(lambda u: 1.0 / (1.0 + u), d, domain=[0.0, R])


def eps_uniforme(R, d, amostras=200_001):
    """eps = ||F - P_d||_inf on [0, R], measured numerically."""
    P = polinomio(R, d)
    us = np.linspace(0.0, R, amostras)
    return float(np.abs(1.0 / (1.0 + us) - P(us)).max())


def scores_aprox(x, sigma, d, termo_k0=True):
    """Scores with the polynomial, in exact arithmetic (no CKKS noise)."""
    R = ((x.max() - x.min()) / sigma) ** 2
    K = polinomio(R, d)(matriz_u(x, sigma))
    if termo_k0:
        np.fill_diagonal(K, 1.0)   # the pair (i,i) contributes exactly K(0)=1
    return K.sum(axis=1)


def gap_de_scores(sc, x):
    """Delta_S at the level of VALUES (Remark 4.2)."""
    por_valor = {v: sc[np.flatnonzero(x == v)[0]] for v in np.unique(x)}
    s_moda = por_valor[v_estrela]
    s_rival = max(s for v, s in por_valor.items() if v != v_estrela)
    return s_moda - s_rival


def rho_bernstein(R):
    """Bernstein ellipse parameter for the pole u = -1, mapped from [0,R]."""
    s0 = 1.0 + 2.0 / R
    return s0 + math.sqrt(s0 * s0 - 1.0)


def sigma_teorema_4_6():
    """Solves f_m > f_y + (n - f_y) K(Delta/sigma) exactly for sigma.
    With K(t) = 1/(1+t^2) this gives  sigma < Delta_min * sqrt(Delta_f/(n - f_m)).
    SUFFICIENT condition."""
    return delta_min * math.sqrt(delta_f / (n - f_m))


print(f"\nLimite do Teorema 4.6:  sigma < {sigma_teorema_4_6():.5f}")
print("(suficiente, nao necessario: ver varrimento abaixo)")

## 2. Plaintext sweep over `(sigma, d)`

For each pair the sweep reports:

- `R = (Delta_max/sigma)^2`: the approximation domain (Section 4.6)
- `eps`: the measured uniform error
- `Delta_S`: the exact score gap (Section 4.8)
- whether `n*eps < Delta_S/2`, the condition of Theorem 4.12
- whether the approximate argmax falls in the modal class

The last column shows where the protocol still works although the sufficient
condition fails.

In [ ]:
classe_modal = set(np.flatnonzero(dados == v_estrela).tolist())

print(f"{'sigma':>6} {'d':>4} {'R':>10} {'eps':>10} {'n*eps':>9} "
      f"{'Delta_S':>9} {'cond':>6} {'argmax':>7} {'S(v*)':>8}")
print("-" * 78)

linhas = []
for sigma in (1.0, 2.0, 3.0, 5.0):
    R = (delta_max / sigma) ** 2
    exatos = scores_exatos(dados, sigma)
    dS = gap_de_scores(exatos, dados)
    for d in (15, 31, 63, 127):
        eps = eps_uniforme(R, d)
        ap = scores_aprox(dados, sigma, d)
        ok_cond = n * eps < dS / 2
        ok_argmax = int(np.argmax(ap)) in classe_modal
        linhas.append((sigma, d, R, eps, dS, ok_cond, ok_argmax))
        print(f"{sigma:6.1f} {d:4d} {R:10.1f} {eps:10.2e} {n*eps:9.3f} "
              f"{dS:9.3f} {'SIM' if ok_cond else 'nao':>6} "
              f"{'SIM' if ok_argmax else 'NAO':>7} {ap[0]:8.4f}")

print(f"\nScore exato da moda (sigma=1): {scores_exatos(dados, 1.0)[0]:.4f}")
print(f"Score exato da moda (sigma=3): {scores_exatos(dados, 3.0)[0]:.4f}")

### Checking the Chebyshev rate

Proposition 4.23 gives the exact truncation error of the auxiliary kernel,
`||F - T_d F|| = 2 rho^(-d) / (sqrt(R+1) (rho-1))`, where `rho` is the parameter of the
Bernstein ellipse through the pole of `F`. This cell checks that the measured decay has that rate.


In [ ]:
for sigma in (1.0, 3.0):
    R = (delta_max / sigma) ** 2
    rho = rho_bernstein(R)
    graus = np.arange(8, 80, 4)
    eps = np.array([eps_uniforme(R, int(d), 50_001) for d in graus])

    # slope of log(eps) against d;  predicted: -log(rho)
    declive = np.polyfit(graus, np.log(eps), 1)[0]

    print(f"sigma = {sigma}   R = {R:.1f}   rho = {rho:.4f}")
    print(f"  -log(rho) previsto : {-math.log(rho):+.5f}")
    print(f"  declive medido     : {declive:+.5f}")
    print(f"  racio              : {declive / -math.log(rho):.3f}")
    print(f"  d para eps < 0.01  : {math.ceil(math.log(1/0.01)/math.log(rho))}\n")

## 3. FHE run: per-candidate traversal

This is the reference implementation of Section 4.13.3 of the dissertation. The loop runs
over candidates `i`, and each iteration performs the four steps listed there:

1. **Broadcast.** A one-hot mask on slot `i`, then `rotate-and-add` by powers of two
   until `x_i` is replicated in all `N` slots.
2. **Difference and square.** `enc - centre`, then one squaring, which gives
   `u_ij = ((x_i - x_j)/sigma)^2` in every slot at once.
3. **Chebyshev evaluation.** `P_d(u_ij)` by Paterson-Stockmeyer.
4. **Masked reduction.** The validity mask `M`, then `rotate-and-add` to sum the
   `N` slots.

This gives the `Theta(n sqrt d)` products and `O(n log n)` rotations of Proposition 4.21.

Four choices keep the run within the guarantee of Theorem 4.16:

- `sigma = 3.0` and `d = 31`, so that `n*eps = 0.005` stays below `Delta_S/2 = 3.98`.
  With `sigma = 1.0` and `d = 15` the condition fails (14.09 against 4.47).
- Padding slots hold a value inside `[0, R]`. A padding value of `0.0` falls outside it.
- Rotation keys are the `log2(N)` powers of two, none of them tied to a candidate.
- The circuit is provisioned with exactly the depth it consumes, 10 levels.

No alignment rotation `EvalRotate(x, i)` is needed: the doubling of step 1 fills every
slot from any starting position. This keeps the key count at the `O(log n)` of
Proposition 4.21 instead of `O(n)`.


In [ ]:
import openfhe
import os
import tempfile
import time

# ---- parameters chosen from the sweep above -------------------------------
SIGMA = 3.0
GRAU = 31
# ---------------------------------------------------------------------------

x_esc = dados / SIGMA
R_fhe = (x_esc.max() - x_esc.min()) ** 2

# Proposition 4.23: exact Chebyshev error for F(u) = 1/(1+u) on [0, R].
s0 = 1.0 + 2.0 / R_fhe
rho = s0 + math.sqrt(s0 * s0 - 1.0)
cota_interp = 4.0 * rho ** -GRAU / (math.sqrt(R_fhe + 1.0) * (rho - 1.0))
dS_ex = gap_de_scores(scores_exatos(dados, SIGMA), dados)
d_cert = math.ceil(math.log(8 * n / (math.sqrt(R_fhe + 1) * (rho - 1) * dS_ex)) / math.log(rho))

print(f"sigma = {SIGMA}   grau = {GRAU}   R = {R_fhe:.2f}   rho = {rho:.4f}")
print(f"cota do interpolante = {cota_interp:.3e}   medido = {eps_uniforme(R_fhe, GRAU):.3e}")
print(f"grau minimo certificado (Teorema 4.12 com a Proposicao 4.23): d = {d_cert}")

# Measured depth (Remark 4.22): mask (1) + square (1)
# + Chebyshev on [0, R] (ceil(log2(d+1)) + 2) + mask (1).
# Provision exactly this much: one more level crosses the ring threshold
# and doubles the cost of everything (Table 4.2).
prof = 1 + 1 + (math.ceil(math.log2(GRAU + 1)) + 2) + 1

params = openfhe.CCParamsCKKSRNS()
params.SetMultiplicativeDepth(prof)
params.SetScalingModSize(50)
params.SetBatchSize(N)

cc = openfhe.GenCryptoContext(params)
cc.Enable(openfhe.PKESchemeFeature.PKE)
cc.Enable(openfhe.PKESchemeFeature.LEVELEDSHE)
cc.Enable(openfhe.PKESchemeFeature.ADVANCEDSHE)
print(f"profundidade provisionada = {prof}   anel = 2^{cc.GetRingDimension().bit_length() - 1}")

keys = cc.KeyGen()
cc.EvalMultKeyGen(keys.secretKey)

# Steps 1 and 4 use rotate-and-add by powers of two, in one direction only.
# No key depends on the candidate (Proposition 4.21).
rot_keys = [-(2 ** k) for k in range(int(math.log2(N)))]
cc.EvalRotateKeyGen(keys.secretKey, rot_keys)


def material_de_chaves_MB(contexto, chave):
    """Serialized size of the evaluation keys of this secret key.
    Without the tag, OpenFHE would serialize the keys of every context in the process."""
    pasta = tempfile.mkdtemp()
    fm, fr = os.path.join(pasta, "m"), os.path.join(pasta, "r")
    tag = chave.secretKey.GetKeyTag()
    contexto.SerializeEvalMultKey(fm, openfhe.BINARY, tag)
    contexto.SerializeEvalAutomorphismKey(fr, openfhe.BINARY, tag)
    return (os.path.getsize(fm) + os.path.getsize(fr)) / 1e6


print(f"chaves de rotacao: {len(rot_keys)}   "
      f"material de chaves: {material_de_chaves_MB(cc, keys):.0f} MB")

# Padding slots must hold a value INSIDE [0, R]. With 0.0, the (real, padding)
# pairs give u = x^2 far above R and the Chebyshev polynomial blows up, which
# ruins the precision of the whole ciphertext because the CKKS scale is global.
PAD = float(x_esc[0])

enc = cc.Encrypt(keys.publicKey,
                 cc.MakeCKKSPackedPlaintext(list(x_esc) + [PAD] * (N - n)))


def kernel_func(u):
    return 1.0 / (1.0 + max(0.0, u))


def mascara(vals, ct=None):
    """Plaintext at the ciphertext's level (avoids a level mismatch in EvalMult)."""
    if ct is not None:
        try:
            return cc.MakeCKKSPackedPlaintext(vals, 1, ct.GetLevel(), None, N)
        except Exception:
            pass
    return cc.MakeCKKSPackedPlaintext(vals)


contagem = {"rotacoes": 0, "quadrados": 0, "mascaras": 0, "chebyshev": 0}


def rotate_and_add(ct):
    """Doubling by powers of two: sums or broadcasts the N slots in log2(N) steps."""
    step = 1
    while step < N:
        ct = cc.EvalAdd(ct, cc.EvalRotate(ct, -step))
        contagem["rotacoes"] += 1
        step *= 2
    return ct


def difundir(ct, i):
    """Step 1: extracts x_i and replicates it across all slots."""
    m = [0.0] * N
    m[i] = 1.0
    contagem["mascaras"] += 1
    return rotate_and_add(cc.EvalMult(ct, mascara(m, ct)))


def somar_validos(ct):
    """Step 4: zeroes the padding slots and sums the rest."""
    m = [1.0] * n + [0.0] * (N - n)
    contagem["mascaras"] += 1
    return rotate_and_add(cc.EvalMult(ct, mascara(m, ct)))


# ---- per-candidate loop (Section 4.13.3) ----------------------------------
enc_scores = []
t0 = time.perf_counter()
for i in range(n):
    centro = difundir(enc, i)                                         # step 1
    diff = cc.EvalSub(enc, centro)
    d2 = cc.EvalMult(diff, diff)                                      # step 2
    contagem["quadrados"] += 1
    kv = cc.EvalChebyshevFunction(kernel_func, d2, 0.0, R_fhe, GRAU)  # step 3
    contagem["chebyshev"] += 1
    enc_scores.append(somar_validos(kv))                              # step 4
    if i % 10 == 0:
        print(f"  [servidor] candidato {i}/{n}")
t_servidor = time.perf_counter() - t0

print(f"\ncontagem medida : {contagem}")
print(f"niveis consumidos: {enc_scores[0].GetLevel()} de {prof}")
print(f"tempo do servidor: {t_servidor:.1f} s  ({t_servidor / n:.2f} s por candidato)")

# ---- n decryptions (Proposition 4.21: Theta(n) ciphertexts returned) -----
scores_fhe = np.empty(n)
for i, ct in enumerate(enc_scores):
    pt = cc.Decrypt(keys.secretKey, ct)
    pt.SetLength(1)
    scores_fhe[i] = pt.GetRealPackedValue()[0]


In [ ]:
# ============================================================
# 4. COMPARISON
# ============================================================
exatos = scores_exatos(dados, SIGMA)
# termo_k0=False: the pair (i,i) goes through the polynomial, as in the FHE circuit
aprox = scores_aprox(dados, SIGMA, GRAU, termo_k0=False)

print(f"{'i':>4} {'x_i':>8} {'exato':>10} {'polinomio':>11} {'FHE':>10} {'|dif|':>9}")
print("-" * 56)
for i in np.argsort(scores_fhe)[::-1][:12]:
    print(f"{i:4d} {dados[i]:8.2f} {exatos[i]:10.4f} {aprox[i]:11.4f} "
          f"{scores_fhe[i]:10.4f} {abs(exatos[i]-scores_fhe[i]):9.4f}")

dS = gap_de_scores(exatos, dados)
eps = eps_uniforme(R_fhe, GRAU)
eta = float(np.abs(aprox - scores_fhe).max())   # observed CKKS noise

print(f"\n--- Teorema 4.16 ---------------------------------------")
print(f"Delta_S            = {dS:.4f}")
print(f"n*eps              = {n*eps:.4f}")
print(f"eta observado      = {eta:.2e}")
print(f"n*eps + eta        = {n*eps + eta:.4f}")
print(f"Delta_S / 2        = {dS/2:.4f}")
print(f"condicao satisfeita: {n*eps + eta < dS/2}")

melhor = int(np.argmax(scores_fhe))
arg_fhe = set(np.flatnonzero(
    scores_fhe > scores_fhe.max() - 10 * max(eta, 1e-9)).tolist())

print(f"\n--- Resultado ------------------------------------------")
print(f"moda recuperada    = {dados[melhor]}   (correto: {v_estrela})")
print(f"classe modal M     = {len(classe_modal)} indices")
print(f"argmax dentro de M : {arg_fhe <= classe_modal}")
print(f"desvio entre copias: {np.ptp(scores_fhe[list(classe_modal)]):.3e}")
print("(Remark 4.17: o ruido CKKS quebra os empates exatos)")

## 5. Block packing (Proposition 4.27)

The per-candidate loop uses `N = 64` of 16 384 slots. Proposition 4.27 uses the rest:
the client, who holds the data in the clear, prepares two encodings in blocks of `N` slots,
and the server runs a single Chebyshev evaluation for all `n` candidates.

- `D` repeats the dataset in every block;
- `C` holds, across all of block `b`, the value of candidate `b`.

The server computes `(D - C)^2`, evaluates `P_d`, applies the mask and runs the
`rotate-and-add` of step 4. Each slot accumulates the `N - 1` slots before it, and the window
ending at the last slot of block `b` is exactly block `b`, so that slot holds the score of
candidate `b`. The other slots mix neighbouring blocks and are discarded. The client pays
for this by sending two ciphertexts instead of one.


In [ ]:
prof_b = prof - 1                          # without the broadcast mask of step 1
lote = 2 ** math.ceil(math.log2(n * N))    # n blocks of N slots, rounded up
B = lote // N

pb = openfhe.CCParamsCKKSRNS()
pb.SetMultiplicativeDepth(prof_b)
pb.SetScalingModSize(50)
pb.SetBatchSize(lote)
ccb = openfhe.GenCryptoContext(pb)
for f in (openfhe.PKESchemeFeature.PKE, openfhe.PKESchemeFeature.LEVELEDSHE,
          openfhe.PKESchemeFeature.ADVANCEDSHE):
    ccb.Enable(f)
kb = ccb.KeyGen()
ccb.EvalMultKeyGen(kb.secretKey)
ccb.EvalRotateKeyGen(kb.secretKey, rot_keys)

# ---- client: two block encodings -----------------------------------------
D, Cb, Mb = [], [], []
for b in range(B):
    D += list(x_esc) + [PAD] * (N - n)
    Cb += [float(x_esc[b]) if b < n else PAD] * N
    Mb += ([1.0] * n + [0.0] * (N - n)) if b < n else [0.0] * N

eD = ccb.Encrypt(kb.publicKey, ccb.MakeCKKSPackedPlaintext(D))
eC = ccb.Encrypt(kb.publicKey, ccb.MakeCKKSPackedPlaintext(Cb))

# ---- server: one pipeline for all candidates ------------------------------
t0 = time.perf_counter()
dif_b = ccb.EvalSub(eD, eC)
kv_b = ccb.EvalChebyshevFunction(kernel_func, ccb.EvalMult(dif_b, dif_b), 0.0, R_fhe, GRAU)
y_b = ccb.EvalMult(kv_b, ccb.MakeCKKSPackedPlaintext(Mb, 1, kv_b.GetLevel(), None, lote))
step, rot_b = 1, 0
while step < N:
    y_b = ccb.EvalAdd(y_b, ccb.EvalRotate(y_b, -step))
    rot_b += 1
    step *= 2
t_bloco = time.perf_counter() - t0

# ---- client: one decryption, score of b in the last slot of block b ------
pt = ccb.Decrypt(kb.secretKey, y_b)
pt.SetLength(lote)
v = np.array(pt.GetRealPackedValue())
scores_bloco = np.array([v[b * N + N - 1] for b in range(n)])
eta_b = float(np.abs(aprox - scores_bloco).max())
i_b = int(np.argmax(scores_bloco))

print(f"anel = 2^{ccb.GetRingDimension().bit_length() - 1}   blocos = {B}   "
      f"niveis consumidos = {y_b.GetLevel()} de {prof_b}")
print(f"avaliacoes de Chebyshev: 1   rotacoes: {rot_b}   "
      f"material de chaves: {material_de_chaves_MB(ccb, kb):.0f} MB")
print(f"tempo do servidor: {t_bloco:.2f} s   (por candidato: {t_servidor:.1f} s, "
      f"fator {t_servidor / t_bloco:.0f})")
print(f"eta observado      : {eta_b:.2e}")
print(f"moda recuperada    : {dados[i_b]}   argmax dentro de M: {i_b in classe_modal}")
